RAG - Retrieval Augmented Generation (Извличащо усилено генериране)

In [4]:
!pip install -q chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [1]:
import json

from pprint import pprint
from pydantic import BaseModel, Field
from typing import List

def print_response(response):
  print(f"Response id: {response.id}")
  print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached); Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
  pprint(response.output)

  print()
  print(f"{'-' * 20} [Text] {'-' * 20}")
  print(response.output_text)


In [2]:
courses = [
    {
        "id": "ai_ass_dev",
        "name": "AI-Assisted Development",
        "description": "\"AI-Assisted Development\" is a practically oriented course that changes the way developers write and maintain code. In modern development, AI assistants have become a necessity for teams seeking greater speed, quality, and consistency throughout the entire code lifecycle. The course is designed for programmers who want to multiply their productivity by integrating GitHub Copilot, Cursor, Augment Code, and Claude Code directly into their daily workflow. During the training, students will master effective prompt techniques, autocompletion and refactoring with Copilot, multi-file changes and agentic flows in Cursor, test generation, documentation, and safe migrations with Augment Code, as well as large-scale edits and code review with large context in Claude Code."
    },
    {
        "id": "ai_int_dev",
        "name": "AI Integrations for Developers",
        "description": "\"AI Integrations for Developers\" is a practical course that will teach students to embed AI functionalities directly into their applications: integrations with OpenAI, Anthropic (Claude), and OpenRouter, embeddings, vector databases, and RAG for semantic search, multimodal services (images, speech, audio, video), and model fine-tuning. The emphasis is on reliability, cost, and security: streaming and structured outputs, retry/timeout/rate limits, telemetry, metrics, data protection, and fallback strategies for non-deterministic responses. Each topic includes an exercise, and the finale features a Workshop: Real-Life Project and a Workshop: Local AI (working with local models such as Ollama/vLLM). Result: confident, production-ready AI integrations."
    },
    {
        "id": "ai_ag_work_dev",
        "name": "AI Agents & Workflows for Developers",
        "description": "\"AI Agents & Workflows for Developers\" is a practical course on designing and implementing agents and orchestrations in real-world applications: n8n for visual automations and integrations, LangChain Agents & Tools for tool-augmented agents, Memory & Human-in-the-Loop for state management and approvals, and LangGraph for reliable multi-agent systems. The emphasis is on reliability, observability, and safety: routing and retries, timeouts, guardrails, logs/metrics and cost tracking, as well as HITL processes for quality. Each topic includes an exercise, followed by Exam Preparation, and the finale is a Workshop: Real-Life Project with end-to-end construction of an agent architecture. Result: robust, extensible, and controllable AI workflows in a production environment."
    },
    {
        "id": "csharp_adv",
        "name": "C# Advanced",
        "description": "The \"C# Advanced\" course builds upon the skills for working with the C# language and the .NET platform, covering more complex concepts typical of the language. During the course, students will learn to create and work with linear data structures. They will expand their knowledge of working with arrays by learning to work with multidimensional arrays or matrices. They will have the opportunity to familiarize themselves with the Generics concept — creating template classes and methods. They will solve algorithmic problems (problem-solving skills) and work with streams, files, and directories. Attention will be given to the functional programming paradigm, as well as its primary tool — LINQ for processing data streams. The development environment used by the training team will be Microsoft Visual Studio 2022, but each student is free to use their preferred tools. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "csharp_oop",
        "name": "C# OOP",
        "description": "The \"C# OOP\" course will teach students the principles of object-oriented programming (OOP), how to work with classes and objects, use object-oriented modeling, and build class hierarchies. The fundamental principles of OOP will be studied, such as abstraction (interfaces and abstract classes), encapsulation, inheritance, and polymorphism. The course will delve into the most commonly used design patterns (creational, structural, and behavioral design patterns). Participants will become familiar with the SOLID principles of object-oriented software design. Various debugging techniques will be covered. Students will learn how to create and use decorators. Attention will be given to component testing (writing unit tests) and the concept of Test-Driven Development (TDD). Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "js_adv",
        "name": "JS Advanced",
        "description": "In the \"JS Advanced\" course, students will gain in-depth knowledge of the JavaScript language, including syntax fundamentals, working with arrays, matrices, objects, classes, and writing functions. They will study more complex concepts such as function context, explicit binding, and the event loop. The course will develop their algorithmic thinking. Upon successful completion of this course, they will be able to work with the DOM tree, perform manipulations on it, and work with events. The functional and OOP approaches to programming with JavaScript will be covered, studying concepts such as inheritance, object composition, and the prototype chain. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "js_appl",
        "name": "JS Applications",
        "description": "In the \"JS Applications\" course, students will learn what HTTP requests are and how to use them. They will become familiar with REST services, what a BaaS (Backend as a Service) is and how to work with it, what asynchronous code means (Promises, using async/await), and what Templating and Routing are. During the course, they will create a Single Page Application using the techniques learned from previous lectures. They will understand the architecture of an application and how to properly structure their applications. Toward the end of the course, they will explore various design patterns and their practical applications, create their own web components using the Web Components standard, and set up a Webpack environment from scratch. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "djng_basics",
        "name": "Django Basics",
        "description": "In the Django Basics course, we will lay the foundations of web programming with Python and Django. We will explore how networks actually work, what HTTP is, and what the fundamental principles of web development are. The course will cover the core concepts of the MTV (Model–Template–View) architecture, such as Function-Based Views and Class-Based Views, and in addition to these, we will use forms (Form and ModelForm) for application development, work with media files, and store data in PostgreSQL. The training includes practical exercises (labs) and workshops for building complete, fully functional Django web applications."
    },
    {
        "id": "py_adv",
        "name": "Python Advanced",
        "description": "The \"Python Advanced\" course builds upon the skills for working with the Python language, covering more complex concepts typical of the language. During the course, students will learn to create linear data structures, solve algorithmic problems (problem-solving skills), and work with stacks and queues, tuples and sets, matrices (multidimensional lists), as well as files and directories. Attention will be given to the functional programming paradigm. Recursive functions and functions with multiple arguments will be explored in greater depth. The development environment used by the training team will be PyCharm, but each student is free to use their preferred tools. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    }
]

In [5]:
from chromadb import PersistentClient

PATH_TO_CHROMA = "/content/chromadb"
chroma_client = PersistentClient(PATH_TO_CHROMA)

In [6]:
chroma_collection = chroma_client.get_or_create_collection(name="softuni-courses")

In [7]:
ids = [c["id"] for c in courses]
metadatas = [{ "name": c["name"] } for c in courses]
documents = [c["description"] for c in courses]

chroma_collection.add(ids=ids, metadatas=metadatas, documents=documents)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 95.6MiB/s]


In [8]:
print(ids)
print(metadatas)
print(documents)

['ai_ass_dev', 'ai_int_dev', 'ai_ag_work_dev', 'csharp_adv', 'csharp_oop', 'js_adv', 'js_appl', 'djng_basics', 'py_adv']
[{'name': 'AI-Assisted Development'}, {'name': 'AI Integrations for Developers'}, {'name': 'AI Agents & Workflows for Developers'}, {'name': 'C# Advanced'}, {'name': 'C# OOP'}, {'name': 'JS Advanced'}, {'name': 'JS Applications'}, {'name': 'Django Basics'}, {'name': 'Python Advanced'}]
['"AI-Assisted Development" is a practically oriented course that changes the way developers write and maintain code. In modern development, AI assistants have become a necessity for teams seeking greater speed, quality, and consistency throughout the entire code lifecycle. The course is designed for programmers who want to multiply their productivity by integrating GitHub Copilot, Cursor, Augment Code, and Claude Code directly into their daily workflow. During the training, students will master effective prompt techniques, autocompletion and refactoring with Copilot, multi-file change

In [10]:
from google.colab import userdata
from openai import OpenAI

openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

In [12]:

class LookupCourseReq(BaseModel):
    """
    Searches the course catalog using semantic similarity to find courses
    matching the user's intent. Use this tool when a user asks about
    available courses, course content, prerequisites, or recommendations.
    Returns the top matching courses ranked by relevance.
    """
    query: str = Field(description="A natural language search query describing the desired course topic, skill, or technology (e.g. 'object-oriented programming with C#', 'building web apps with Django', 'AI agents and workflows')")
    top_n: int = Field(default=5, gt=1, lt=20, description="The maximum number of matching courses to return, ordered by relevance")

class CourseInfo(BaseModel):
    name: str
    description: str


def lookup_course(req: LookupCourseReq) -> List[CourseInfo]:
    lookup_result = chroma_collection.query(
        query_texts=[req.query],
        n_results=req.top_n
    )
    print(lookup_result)

    results_count = len(lookup_result["ids"][0])
    return [CourseInfo(name=lookup_result["metadatas"][0][i]["name"], description=lookup_result["documents"][0][i]) for i in range(results_count)]

In [13]:
system_prompt = "You are a customer support agent for SoftUni (Software University) - a leading tech education provider offering programming courses, professional training, and career development programs for aspiring and experienced developers." \
                "Your goal is to help students and prospective students resolve their questions quickly, accurately, and with genuine care for their learning journey." \
                "Use the \"lookup_course\" tool whenever a student asks about course content, recommendations, comparisons, prerequisites, or anything that requires knowledge of the course catalog. Always search before answering course-specific questions - do not rely on memory alone."

search_for_ai_related_course_prompt = "Hello, I am trying to find an AI-related course. I want to integrate AI into my professional life as a software developer."

In [15]:
lookup_course_tool_schema = LookupCourseReq.model_json_schema()
print(lookup_course_tool_schema)
tools = [
    {
        "type": "function",
        "name": "lookup_course",
        "descrtiption": lookup_course_tool_schema["description"],
        "parameters": {
            "type": "object",
            "properties": lookup_course_tool_schema["properties"],
            "required": list(lookup_course_tool_schema["properties"].keys()),
            "additionalProperties": False
        },
        "strict": True
    }
]

tool_call_handlers = {
    "lookup_course": lambda args: [c.model_dump() for c in lookup_course(LookupCourseReq(**args))]
}

{'description': "Searches the course catalog using semantic similarity to find courses\nmatching the user's intent. Use this tool when a user asks about\navailable courses, course content, prerequisites, or recommendations.\nReturns the top matching courses ranked by relevance.", 'properties': {'query': {'description': "A natural language search query describing the desired course topic, skill, or technology (e.g. 'object-oriented programming with C#', 'building web apps with Django', 'AI agents and workflows')", 'title': 'Query', 'type': 'string'}, 'top_n': {'default': 5, 'description': 'The maximum number of matching courses to return, ordered by relevance', 'exclusiveMaximum': 20, 'exclusiveMinimum': 1, 'title': 'Top N', 'type': 'integer'}}, 'required': ['query'], 'title': 'LookupCourseReq', 'type': 'object'}


In [16]:
def execute_tools(response):
    tool_calls = [item for item in response.output if item.type == "function_call"]

    result = []
    for tool_call in tool_calls:
        args = json.loads(tool_call.arguments)
        if tool_call.name not in tool_call_handlers:
            raise Exception("Unknown tool!")

        output = tool_call_handlers[tool_call.name](args)
        result.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(output)
        })

    return result

In [17]:
conversation = [
    { "role": "developer", "content": system_prompt },
    { "role": "user", "content": search_for_ai_related_course_prompt }
]

search_for_ai_related_course_response = openai_client.responses.create(
    model="gpt-5-nano",
    input=conversation,
    tools=tools
)

In [18]:
print_response(search_for_ai_related_course_response)

Response id: resp_0d4de7c67657a5df0069df985794e48197ad655ea4e7e86087
Input tokens: 255 (0 cached); Output tokens: 263 (192 reasoning)
[ResponseReasoningItem(id='rs_0d4de7c67657a5df0069df9858389c819788fab3eacd2463b3', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseFunctionToolCall(arguments='{"query":"AI for software developers", "top_n":5}', call_id='call_yefSHdpwp70NYT5pV4pN8BI3', name='lookup_course', type='function_call', id='fc_0d4de7c67657a5df0069df985a45488197b01b20cff989522a', namespace=None, status='completed')]

-------------------- [Text] --------------------



In [19]:
conversation.extend(search_for_ai_related_course_response.output)

In [24]:
print(search_for_ai_related_course_response.output)

[ResponseReasoningItem(id='rs_0d4de7c67657a5df0069df9858389c819788fab3eacd2463b3', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"query":"AI for software developers", "top_n":5}', call_id='call_yefSHdpwp70NYT5pV4pN8BI3', name='lookup_course', type='function_call', id='fc_0d4de7c67657a5df0069df985a45488197b01b20cff989522a', namespace=None, status='completed')]


In [20]:
print(conversation)

[{'role': 'developer', 'content': 'You are a customer support agent for SoftUni (Software University) - a leading tech education provider offering programming courses, professional training, and career development programs for aspiring and experienced developers.Your goal is to help students and prospective students resolve their questions quickly, accurately, and with genuine care for their learning journey.Use the "lookup_course" tool whenever a student asks about course content, recommendations, comparisons, prerequisites, or anything that requires knowledge of the course catalog. Always search before answering course-specific questions - do not rely on memory alone.'}, {'role': 'user', 'content': 'Hello, I am trying to find an AI-related course. I want to integrate AI into my professional life as a software developer.'}, ResponseReasoningItem(id='rs_0d4de7c67657a5df0069df9858389c819788fab3eacd2463b3', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),

In [21]:
tool_call_results = execute_tools(search_for_ai_related_course_response)
conversation.extend(tool_call_results)
pprint(tool_call_results)

{'ids': [['ai_ass_dev', 'ai_int_dev', 'ai_ag_work_dev', 'csharp_adv', 'js_adv']], 'embeddings': None, 'documents': [['"AI-Assisted Development" is a practically oriented course that changes the way developers write and maintain code. In modern development, AI assistants have become a necessity for teams seeking greater speed, quality, and consistency throughout the entire code lifecycle. The course is designed for programmers who want to multiply their productivity by integrating GitHub Copilot, Cursor, Augment Code, and Claude Code directly into their daily workflow. During the training, students will master effective prompt techniques, autocompletion and refactoring with Copilot, multi-file changes and agentic flows in Cursor, test generation, documentation, and safe migrations with Augment Code, as well as large-scale edits and code review with large context in Claude Code.', '"AI Integrations for Developers" is a practical course that will teach students to embed AI functionalities

In [22]:
continue_after_tool_calls_response = openai_client.responses.create(
    model="gpt-5-nano",
    input=conversation,
    tools=tools
)

In [23]:
print_response(continue_after_tool_calls_response)

Response id: resp_0d4de7c67657a5df0069dfa0a289f48197b9210b6445c25ae9
Input tokens: 1430 (0 cached); Output tokens: 1354 (832 reasoning)
[ResponseReasoningItem(id='rs_0d4de7c67657a5df0069dfa0a379508197a062abaecc09d761', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_0d4de7c67657a5df0069dfa0aade0081978e8d3c275399b5de', content=[ResponseOutputText(annotations=[], text='Great choice — AI is increasingly essential for software developers. Based on your goal of integrating AI into your professional life, I’d recommend starting with these AI-focused SoftUni courses:\n\n- AI-Assisted Development\n  - What you’ll learn: Boost productivity by integrating AI copilots and tools (GitHub Copilot, Cursor, Augment Code, Claude Code) directly into your workflow. Master prompts, automated refactoring, multi-file changes, agent-like workflows, test generation, documentation, safe migrations, and large-context code reviews.\n  - Why it fits